# Лекция 5. Исключения и тестирование в Python

### Темы
1. Ошибки и исключения
2. `try / except / else / finally`
3. `raise` и повторное возбуждение исключения
4. Собственные исключения
5. `assert`
6. Идея модульного тестирования
7. `unittest`
8. Проверка исключений в тестах
9. Подготовка данных: `setUp`


Лекция прочитана Иваном Кисляковым. Разрешено к использованию в некоммерческих образовательных целях.

## 1. Ошибка в программе: что именно произошло?

В Python полезно различать:

- **синтаксическую ошибку** — Python не может разобрать программу;
- **исключение** — программа синтаксически корректна, но во время выполнения возникла проблема;
- **логическую ошибку** — программа выполняется, но выдаёт неправильный результат.

Когда возникает исключение, Python строит **traceback** — цепочку вызовов, которая показывает, где именно возникла ошибка.

In [1]:
# Синтаксическая ошибка.
# Раскомментируйте две строки ниже и выполните ячейку.

if 10 > 5
     print("Да")

SyntaxError: expected ':' (2626895463.py, line 4)

In [3]:
# IndexError: индекс существует синтаксически, но элемента с таким индексом нет.
numbers = [10, 20, 30]

# Раскомментируйте:
num = int(input())
numbers[num]

 10


IndexError: list index out of range

In [4]:
# Несколько распространённых исключений.
examples = {
    "ZeroDivisionError": "1 / 0",
    "NameError": "print(unknown_name)",
    "KeyError": "{'a': 1}['b']",
    "TypeError": "'10' + 5",
    "ValueError": "int('abc')",
}

examples

{'ZeroDivisionError': '1 / 0',
 'NameError': 'print(unknown_name)',
 'KeyError': "{'a': 1}['b']",
 'TypeError': "'10' + 5",
 'ValueError': "int('abc')"}

### Как читать traceback

Смотрите **снизу вверх**:

1. последняя строка — тип исключения и сообщение;
2. строка выше — место, где исключение возникло;
3. ещё выше — цепочка вызовов функций, приведшая к ошибке.

Для отладки обычно особенно важны:
- тип исключения;
- текст сообщения;
- номер строки;
- значения переменных в этой точке.

In [5]:
def divide(a, b):
    return a / b

def calculate():
    return divide(10, 0)

# Раскомментируйте и прочитайте traceback снизу вверх:
calculate()

ZeroDivisionError: division by zero

## 2. `try / except`

Если исключение ожидаемо и программа умеет на него реагировать, используется конструкция:

```python
try:
    # потенциально опасный код
except SomeException:
    # обработка конкретной ошибки
```

Лучше перехватывать **конкретные исключения**, а не вообще все подряд.

In [9]:
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return "На ноль делить нельзя"

a = safe_divide(10, 2)
b = safe_divide(10, 0)
c = safe_divide(10, "ab")
print(a, b)

TypeError: unsupported operand type(s) for /: 'int' and 'str'

### Несколько `except`

Python выбирает первый подходящий обработчик.

От более конкретных исключений обычно идут к более общим.

In [10]:
def get_element(data, index):
    try:
        return data[index]
    except IndexError:
        return "Индекс вне диапазона"
    except TypeError:
        return "Объект нельзя индексировать таким способом"

print(get_element([10, 20, 30], 1))

print(get_element([10, 20, 30], 10))

print(get_element([10, 20, 30], "a"))


20
Индекс вне диапазона
Объект нельзя индексировать таким способом


In [20]:
def get_element(data, index):
    try:
        1 / index
        return data[index]
    except IndexError:
        return "Индекс вне диапазона"
    except TypeError:
        return "Объект нельзя индексировать таким способом"
    except:
        return "непонятно что произошло"

print(get_element([10, 20, 30], 1))

print(get_element([10, 20, 30], 10))

print(get_element([10, 20, 30], "a"))

print(get_element([10, 20, 30], 0))






20
Индекс вне диапазона
Объект нельзя индексировать таким способом
непонятно что произошло


### Получение объекта исключения

Конструкция `except ... as e` даёт доступ к сообщению исключения.

```python
except ValueError as e:
    print(type(e).__name__, e)
```

In [23]:
def read_integer(text):
    try:
        return int(text)
    except ValueError as e:
        return f"{type(e).__name__}: {e}"

read_integer("42"), read_integer("сорок два")

(42, "ValueError: invalid literal for int() with base 10: 'сорок два'")

In [31]:
def read_integer(text):
    try:
        return int(text[1])
    except ValueError as e:
        return f"{type(e).__name__}: {e}"
    except:
        return "Нечто"

read_integer("42"), read_integer("сорок два"), read_integer(42)

(2, "ValueError: invalid literal for int() with base 10: 'о'", 'Нечто')

### Почему `except:` почти всегда плохая идея

Пустой `except:` перехватывает не только обычные ошибки программы, но и специальные события вроде `KeyboardInterrupt` и `SystemExit`.

Если нужен общий обработчик обычных исключений, обычно используют:

```python
except Exception as e:
    ...
```

Но и его не стоит применять там, где можно явно перечислить ожидаемые ошибки.

## 3. `else` и `finally`

Полная форма:

```python
try:
    ...
except SomeException:
    ...
else:
    ...
finally:
    ...
```

- `else` выполняется, если в `try` **не возникло исключение**;
- `finally` выполняется **почти всегда**, независимо от того, было исключение или нет.

`finally` полезен для освобождения ресурсов. Для файлов на практике чаще используют `with`.

In [36]:
def divide_with_report(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("Ошибка: деление на ноль")
        return None
    else:
        print("Вычисление прошло успешно")
        return result
    finally:
        print("Выполнится всегда")


divide_with_report(12, 3)
divide_with_report(12, 0)

Вычисление прошло успешно
Выполнится всегда
Ошибка: деление на ноль
Выполнится всегда


In [38]:
try:
    a = 1 / 0
except:
    print("OK")
    try:
        b = 1 / 0
    except:
        print("NOP")

OK
NOP


### Не помещайте в `try` лишний код

Чем меньше код внутри `try`, тем понятнее, **какую именно ошибку мы ожидаем**.

Плохо:

```python
try:
    x = int(text)
    result = 100 / x
    save_result(result)
except ValueError:
    ...
```

Если `save_result` сама бросит `ValueError`, мы можем ошибочно решить, что проблема была в преобразовании строки.

## 4. `raise`: как самостоятельно возбудить исключение

Иногда ошибка определяется логикой нашей функции.

В этом случае функция должна не возвращать специальную строку вроде `"ERROR"` или `"INF"`, а сообщить об ошибке через исключение.

In [39]:
def factorial(n):
    if not isinstance(n, int):
        raise TypeError("n должен быть целым числом")
    if n < 0:
        raise ValueError("Факториал отрицательного числа не определён")

    result = 1
    for value in range(2, n + 1):
        result *= value
    return result

factorial(5)

120

In [43]:
# Раскомментируйте по одному вызову и изучите сообщение.

# factorial(-1)
factorial(2.5)

TypeError: n должен быть целым числом

### Повторное возбуждение исключения

Иногда нужно выполнить действие при ошибке, но не скрывать её:

```python
try:
    ...
except ValueError:
    print("Записываем ошибку в журнал")
    raise
```

Оператор `raise` без аргументов повторно возбуждает текущее исключение.

In [46]:
def parse_age(text):
    try:
        age = int(text)
    except ValueError:
        print("Не удалось преобразовать возраст в целое число")
        raise

    if age < 0:
        raise ValueError("Возраст не может быть отрицательным")

    return age

try:
    parse_age("text")
except ValueError:
    print("Вызов функции завершился ошибкой")

Не удалось преобразовать возраст в целое число
Вызов функции завершился ошибкой


## 5. Собственные исключения

Свои исключения создают наследованием от `Exception`.

**Не стоит** наследовать обычную ошибку приложения непосредственно от `BaseException`: от него также происходят специальные исключения, связанные с завершением программы.

In [47]:
class InvalidScoreError(Exception):
    """Оценка находится вне допустимого диапазона."""


def normalize_score(score):
    if not 0 <= score <= 100:
        raise InvalidScoreError(f"Ожидалось число от 0 до 100, получено {score}")
    return score / 100

normalize_score(75)

0.75

In [48]:
try:
    normalize_score(150)
except InvalidScoreError as e:
    print("Ошибка данных:", e)

Ошибка данных: Ожидалось число от 0 до 100, получено 150


### Когда создавать собственный тип исключения?

Это полезно, если вызывающему коду важно отличить **вашу предметную ошибку** от стандартных ошибок Python.

Например:
- `InvalidScoreError`
- `InsufficientFundsError`
- `WrongPasswordError`
- `RouteNotFoundError`

Смысл типа исключения важнее длинного текста сообщения.

## 6. `assert`

`assert` проверяет утверждение, которое программист считает истинным.

```python
assert condition, "message"
```

Если условие ложно, возникает `AssertionError`.

In [49]:
def average(numbers):
    assert len(numbers) > 0, "Список не должен быть пустым"
    # if len(numbers) <= 0 raise AssertionError("Список не должен быть пустым")
    return sum(numbers) / len(numbers)

average([3, 5, 7])

5.0

In [50]:
average([])

AssertionError: Список не должен быть пустым

### Основные проверки с `assert`

В обычном Python `assert` позволяет проверять логические выражения.

| Проверка | Пример | Что проверяет |
|---|---|---|
| Равенство | `assert a == b` | `a` равно `b` |
| Неравенство | `assert a != b` | `a` не равно `b` |
| Больше | `assert a > b` | `a` больше `b` |
| Меньше | `assert a < b` | `a` меньше `b` |
| Больше или равно | `assert a >= b` | `a` не меньше `b` |
| Меньше или равно | `assert a <= b` | `a` не больше `b` |
| Истинность | `assert condition` | условие истинно |
| Ложность | `assert not condition` | условие ложно |
| Принадлежность | `assert x in collection` | `x` содержится в коллекции |
| Отсутствие | `assert x not in collection` | `x` не содержится в коллекции |
| Тождественность | `assert a is b` | `a` и `b` — один и тот же объект |
| Нетождественность | `assert a is not b` | `a` и `b` — разные объекты |
| Проверка `None` | `assert x is None` | значение равно `None` |
| Проверка не `None` | `assert x is not None` | значение не равно `None` |
| Тип объекта | `assert isinstance(x, int)` | `x` имеет указанный тип |
| Длина | `assert len(items) == 3` | коллекция имеет нужную длину |
| Содержимое строки | `assert "error" in message` | подстрока входит в строку |

Можно добавить собственное сообщение:

```python
assert result == expected, "Получен неправильный результат"
```

Сообщение после запятой полезно, если без него причина падения проверки будет непонятна.


### Важно: `assert` — не полноценная проверка пользовательского ввода

Python можно запустить с оптимизацией (`python -O`), и тогда инструкции `assert` могут быть отключены.

Поэтому:

- `assert` удобно применять для внутренних инвариантов, отладки и тестов;
- для обязательной проверки входных данных используйте `if ...: raise ...`.

In [ ]:
def set_percentage(value):
    if not 0 <= value <= 100:
        raise ValueError("Процент должен быть от 0 до 100")
    return value

set_percentage(90)

# Часть II. Модульное тестирование

## 7. Зачем нужны тесты

Тест — это код, который проверяет другой код.

Хороший набор тестов:
- проверяет обычные случаи;
- проверяет границы;
- проверяет пустые и минимальные значения;
- проверяет неправильные данные;
- проверяет ожидаемые исключения;
- помогает обнаружить регрессию после изменения программы.

Удобная схема теста — **Arrange → Act → Assert**:

1. **Arrange** — подготовить данные;
2. **Act** — вызвать тестируемый код;
3. **Assert** — проверить результат.

In [53]:
def is_even(n):
    return n % 2 != 0

# Arrange
number = 14

# Act
result = is_even(number)

# Assert
assert result is True

AssertionError: 

## 8. Что именно тестировать?

Рассмотрим функцию:

In [54]:
def clamp(value, left, right):
    """Ограничивает value отрезком [left, right]."""
    if left > right:
        raise ValueError("left не может быть больше right")
    return max(left, min(value, right))

Для неё полезны тесты:

| Случай | Пример | Ожидаемый результат |
|---|---:|---:|
| внутри диапазона | `clamp(5, 0, 10)` | `5` |
| левая граница | `clamp(0, 0, 10)` | `0` |
| правая граница | `clamp(10, 0, 10)` | `10` |
| левее диапазона | `clamp(-3, 0, 10)` | `0` |
| правее диапазона | `clamp(20, 0, 10)` | `10` |
| неправильные границы | `clamp(5, 10, 0)` | `ValueError` |

Проверка только «обычного» случая почти ничего не гарантирует.

# Часть III. `unittest`

## 9. Базовая структура

`unittest` входит в стандартную библиотеку Python.

Тесты объединяются в классы — наследники `unittest.TestCase`.

In [57]:
import unittest

class TestClamp(unittest.TestCase):
    def test_inside_interval(self):
        self.assertEqual(clamp(5, 0, 10), 5)

    def test_left_side(self):
        self.assertEqual(clamp(-5, 0, 10), 0)

    def test_right_side(self):
        self.assertEqual(clamp(15, 0, 10), 10)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestClamp)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)














test_inside_interval (__main__.TestClamp.test_inside_interval) ... ok
test_left_side (__main__.TestClamp.test_left_side) ... ok
test_right_side (__main__.TestClamp.test_right_side) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.003s

OK


<unittest.runner.TextTestResult run=3 errors=0 failures=0>

## 10. Основные проверки `unittest`

| Метод | Что проверяет |
|---|---|
| `assertEqual(a, b)` | `a == b` |
| `assertNotEqual(a, b)` | `a != b` |
| `assertTrue(x)` | `bool(x) is True` |
| `assertFalse(x)` | `bool(x) is False` |
| `assertIs(a, b)` | `a is b` |
| `assertIsNot(a, b)` | `a is not b` |
| `assertIsNone(x)` | `x is None` |
| `assertIsNotNone(x)` | `x is not None` |
| `assertIn(a, b)` | `a in b` |
| `assertNotIn(a, b)` | `a not in b` |
| `assertIsInstance(a, b)` | `isinstance(a, b)` |
| `assertGreater(a, b)` | `a > b` |
| `assertLess(a, b)` | `a < b` |
| `assertAlmostEqual(a, b)` | числа почти равны |
| `assertRaises(...)` | возникло нужное исключение |

### Проверка исключений через контекстный менеджер

Иногда правильное поведение функции — **не вернуть значение, а выбросить исключение**.

Например:

```python
def factorial(n):
    if n < 0:
        raise ValueError("n must be non-negative")
```

Для отрицательного `n` функция обязана завершиться с `ValueError`.  
Это тоже нужно уметь тестировать.

В `unittest` для этого есть `assertRaises`.

#### Вариант 1. Через контекстный менеджер

```python
with self.assertRaises(ValueError):
    factorial(-1)
```

Читается это так:

> «Проверь, что при выполнении кода внутри блока `with` возникает `ValueError`».

Тест будет успешным, если:
- возник именно `ValueError`;
- исключение возникло внутри блока `with`.

Тест завершится ошибкой, если:
- исключение вообще не возникло;
- возникло исключение другого типа.

#### Вариант 2. Передать функцию в `assertRaises`

То же самое можно записать короче:

```python
self.assertRaises(ValueError, factorial, -1)
```

Здесь:
- `ValueError` — ожидаемый тип исключения;
- `factorial` — функция;
- `-1` — аргумент функции.

Обе записи эквивалентны, но вариант с `with` обычно удобнее читать, особенно если проверяемый код состоит из нескольких строк.

#### Можно получить само исключение

Контекстный менеджер позволяет сохранить объект исключения:

```python
with self.assertRaises(ValueError) as context:
    factorial(-1)

print(context.exception)
```

Теперь можно проверить не только тип исключения, но и его сообщение:

```python
with self.assertRaises(ValueError) as context:
    factorial(-1)

self.assertIn("non-negative", str(context.exception))
```

Это полезно, когда важно проверить, что функция сообщает пользователю именно нужную причину ошибки.

#### Главное

`assertRaises` проверяет **ожидаемую ошибку**.

Если функция должна выбросить исключение, то возникновение этого исключения означает, что тест **пройден**, а не провален.

In [ ]:
print(isinstance(5,int))

In [58]:
def factorial(n):
    if n < 0:
        raise ValueError("n must be non-negative")
    if not isinstance(n, int):
        raise TypeError("n must be integer")
    if n == 0:
        return 1
    return factorial(n - 1) * n


class TestFactorial(unittest.TestCase):
    def test_zero(self):
        self.assertEqual(factorial(0), 1)

    def test_five(self):
        self.assertEqual(factorial(5), 120)

    def test_negative(self):
        with self.assertRaises(ValueError):
            factorial(-1)

    def test_wrong_type(self):
        self.assertRaises(TypeError, factorial, 1.5)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestFactorial)
unittest.TextTestRunner(verbosity=2).run(suite)

test_five (__main__.TestFactorial.test_five) ... ok
test_negative (__main__.TestFactorial.test_negative) ... ok
test_wrong_type (__main__.TestFactorial.test_wrong_type) ... ok
test_zero (__main__.TestFactorial.test_zero) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.003s

OK


<unittest.runner.TextTestResult run=4 errors=0 failures=0>

### `setUp`: общая подготовка данных

Если нескольким тестам нужны одинаковые исходные данные, их можно создавать перед каждым тестом в `setUp`.

In [59]:
class TestListOperations(unittest.TestCase):
    def setUp(self):
        self.values = [10, 20, 30]

    def test_length(self):
        self.assertEqual(len(self.values), 3)

    def test_contains(self):
        self.assertIn(20, self.values)

    def test_sum(self):
        self.assertEqual(sum(self.values), 60)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestListOperations)
unittest.TextTestRunner(verbosity=2).run(suite)

test_contains (__main__.TestListOperations.test_contains) ... ok
test_length (__main__.TestListOperations.test_length) ... ok
test_sum (__main__.TestListOperations.test_sum) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.003s

OK


<unittest.runner.TextTestResult run=3 errors=0 failures=0>

## 12. Один тест — одна понятная причина падения

Не нужно превращать тест в большую программу.

Лучше несколько коротких тестов:
- проще читать;
- проще понять падение;
- проще поддерживать;
- один сбой не скрывает остальные проверки.

## 13. Не тестируйте реализацию, тестируйте поведение

Если функция должна сортировать список, тест должен проверять результат сортировки, а не количество циклов внутри.

Тесты, слишком сильно привязанные к внутренней реализации, ломаются при любом рефакторинге.

## 14. Тесты должны быть воспроизводимыми

Плохой тест случайно проходит или падает.

Если используется случайность:
- фиксируйте seed;
- или проверяйте свойства результата, а не конкретное случайное значение.

То же относится к текущему времени, сети и внешним сервисам.

## 15. Практика 1. Исключения

Напишите функцию:

```python
def get_discounted_price(price, discount):
    ...
```

Требования:
1. `price` и `discount` должны быть числами;
2. `price >= 0`;
3. `0 <= discount <= 100`;
4. при неверном типе бросить `TypeError`;
5. при неверном значении — `ValueError`;
6. вернуть цену после скидки.

После реализации проверьте:
- обычный случай;
- скидку `0%`;
- скидку `100%`;
- отрицательную цену;
- скидку больше `100`;
- строку вместо числа.

In [ ]:
def get_discounted_price(price, discount):
    # TODO
    pass

## 16. Практика 2. `unittest`

Для `get_discounted_price` создайте класс `TestDiscountedPrice`.

Минимум 6 тестов:
- 3 корректных случая;
- 1 проверка `TypeError`;
- 2 проверки `ValueError`.

Старайтесь использовать `with self.assertRaises(...)`.

In [ ]:
class TestDiscountedPrice(unittest.TestCase):
    # TODO
    pass

## 17. Практика 4. Найдите ошибку по тесту

Функция должна возвращать среднее арифметическое непустого списка.
Для пустого списка должна бросать `ValueError`.

In [ ]:
def mean(numbers):
    # В реализации есть проблема.
    return sum(numbers) / len(numbers)

## 28. Команды запуска `unittest` вне Jupyter

```bash
# Автоматически найти и запустить тесты
python -m unittest discover

# Подробный вывод
python -m unittest -v

# Запустить конкретный файл/модуль
python -m unittest test_example

# Запустить конкретный класс
python -m unittest test_example.TestExample

# Запустить конкретный тест
python -m unittest test_example.TestExample.test_case
```


## 29. Минимальный шаблон `unittest`

```python
import unittest

from module_name import function_name


class TestFunctionName(unittest.TestCase):
    def test_regular_case(self):
        self.assertEqual(function_name(...), ...)

    def test_error_case(self):
        with self.assertRaises(ValueError):
            function_name(...)


if __name__ == "__main__":
    unittest.main()
```
